# ARES 2023-68A — BR Tranche Pricing (our price vs market)

Two model prices for the BR tranche, then compare to the actual market price.

- **Method 1 — expected-cash-flow (our loss view):** promised FRN cash flows,
  principal haircut by MC **E[BR loss]**, discounted at SOFR + credit spread.
- **Method 2 — discount-margin (market convention):** promised cash flows, **no**
  loss haircut, discounted at SOFR + DM.

**Assumptions (all parameters below)**
- Valuation date ~2026-05-25 (Periodic Report 05_25_26).
- BR: $52.5M, spread 170bps, all-in coupon ≈ 5.37%. Quarterly (n=4).
- 3M SOFR = 3.668%, flat forward.
- WAL ~5.5y (reinvestment end + amortization) — parameter.
- Loss = **principal-only haircut** at WAL (ignores loss timing) — simplification.
- Method 1 discounts at credit spread AND haircuts EL → reflects both; documented.

> **Market price:** NOT in the provided data → left as `MARKET_PRICE = None`.
> Not fabricated. The comparison cell activates once it is supplied.

## Parameters

In [ ]:
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt

# --- BR terms ---
FACE       = 52_500_000      # $ BR balance
SPREAD     = 0.0170          # 170 bps over SOFR
SOFR       = 0.03668         # 3M SOFR, flat forward
FREQ       = 4               # quarterly
WAL        = 5.5             # yrs, assumption (~5-6y); parameter
COUPON     = SOFR + SPREAD   # all-in ~5.37% (matches reported 5.367%)

# --- discounting ranges ---
BASE_RHO      = 0.20
M1_SPREAD_BASE= 0.0170                       # Method 1 base credit spread = issue spread
M1_SPREADS    = np.arange(0.0150, 0.0401, 0.0025)   # 150-400 bps
M2_DMS        = np.arange(0.0250, 0.0451, 0.0025)   # 250-450 bps

# --- market price slot (do NOT fabricate) ---
MARKET_PRICE = None          # set to BR price (% of par), e.g. 98.5, when available
print(f'coupon={COUPON:.4%}  WAL={WAL}y  SOFR={SOFR:.3%}')

## Expected loss from Monte Carlo

In [ ]:
mc = pd.read_csv('output/br_montecarlo_results.csv')
EL_BY_RHO = mc.set_index('rho')['E[BR loss]']
EL_BASE   = float(EL_BY_RHO.loc[BASE_RHO])
print('E[BR loss] by rho:'); print(EL_BY_RHO.round(5).to_string())
print(f'\nbase EL (rho={BASE_RHO}) = {EL_BASE:.4%}')

## Pricing function (FRN)

- Promised coupon = (SOFR + coupon_spread)/freq per period.
- Principal = (1 − EL) returned at WAL.
- Discount at (SOFR + disc_spread). Price = % of par.

In [ ]:
def price_frn(disc_spread, el=0.0, wal=WAL, sofr=SOFR, cpn_spread=SPREAD, freq=FREQ):
    n = int(round(wal * freq))
    cf = np.full(n, (sofr + cpn_spread) / freq)   # coupon per $1 face
    cf[-1] += (1 - el)                            # principal (loss-haircut) at WAL
    t = np.arange(1, n + 1)
    df = 1 / (1 + (sofr + disc_spread) / freq) ** t
    return 100 * (cf * df).sum()                  # price, % of par

## Method 1 — expected-cash-flow (loss view)
- Base price: EL at base rho, discount spread = issue spread.
- Sensitivity: EL across the rho grid, and discount spread 150-400 bps.

In [ ]:
m1_base = price_frn(M1_SPREAD_BASE, el=EL_BASE)
print(f'Method 1 base price = {m1_base:.2f}  (EL={EL_BASE:.3%}, disc spread={M1_SPREAD_BASE:.2%})')

# rho sensitivity (EL varies, discount spread fixed at base)
m1_rho = pd.DataFrame({'rho': EL_BY_RHO.index, 'EL': EL_BY_RHO.values,
                       'price': [price_frn(M1_SPREAD_BASE, el=e) for e in EL_BY_RHO.values]})
# discount-spread sensitivity (EL fixed at base)
m1_spr = pd.DataFrame({'disc_spread_bps': (M1_SPREADS*1e4).astype(int),
                       'price': [price_frn(s, el=EL_BASE) for s in M1_SPREADS]})
print('\nprice vs rho (EL):'); print(m1_rho.round(4).to_string(index=False))
print('\nprice vs discount spread:'); print(m1_spr.round(3).to_string(index=False))

## Method 2 — discount-margin (market convention)
- Promised cash flows, no loss haircut, discounted at SOFR + DM.
- DM = comparable BB/BBB CLO mezz spread (parameter, 250-450 bps).

In [ ]:
m2 = pd.DataFrame({'DM_bps': (M2_DMS*1e4).astype(int),
                   'price': [price_frn(dm, el=0.0) for dm in M2_DMS]})
print('Method 2 price vs DM:'); print(m2.round(3).to_string(index=False))
print(f'\nat DM=issue spread ({SPREAD:.2%}) price = {price_frn(SPREAD, el=0):.2f} (~par check)')

## Save model prices

In [ ]:
os.makedirs('output', exist_ok=True)
summary = pd.DataFrame({
    'method': ['M1 loss-view (base)', 'M2 DM=300bps'],
    'price':  [round(m1_base,2), round(price_frn(0.0300, el=0),2)],
    'note':   ['EL={:.2%}, disc=issue spread'.format(EL_BASE), 'promised CFs, DM 300bps'],
})
summary.to_csv('output/br_pricing_summary.csv', index=False)
m1_spr.to_csv('output/br_pricing_method1_spread.csv', index=False)
m2.to_csv('output/br_pricing_method2_dm.csv', index=False)
print(summary.to_string(index=False))
print('\nwrote br_pricing_summary.csv, method1_spread.csv, method2_dm.csv')

## Plots

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13,4))
# M1 price vs discount spread
ax[0].plot(m1_spr['disc_spread_bps'], m1_spr['price'], '-o', color='steelblue')
ax[0].axvline(170, ls='--', color='grey'); ax[0].set_title('Method 1: price vs discount spread (EL fixed)')
ax[0].set_xlabel('discount credit spread (bps)'); ax[0].set_ylabel('price (% par)'); ax[0].grid(alpha=.3)
# M2 price vs DM
ax[1].plot(m2['DM_bps'], m2['price'], '-o', color='firebrick')
ax[1].set_title('Method 2: price vs discount margin'); ax[1].set_xlabel('DM (bps)')
ax[1].set_ylabel('price (% par)'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# Method 1 price vs correlation (loss view)
plt.figure(figsize=(7,4))
plt.plot(m1_rho['rho'], m1_rho['price'], '-o', color='seagreen')
plt.title('Method 1: BR price vs correlation (via E[BR loss])')
plt.xlabel('rho'); plt.ylabel('price (% par)'); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Compare to market

In [ ]:
if MARKET_PRICE is None:
    print('MARKET PRICE NOT PROVIDED — comparison pending.')
    print('Set MARKET_PRICE (% of par, e.g. 98.5) at the top to activate:')
    print('  - richness/cheapness vs Method 1 (loss view)')
    print('  - implied DM from Method 2 that matches market')
else:
    print(f'market price      = {MARKET_PRICE:.2f}')
    print(f'M1 loss-view      = {m1_base:.2f}   diff = {m1_base-MARKET_PRICE:+.2f}')
    # implied DM: find DM whose Method-2 price = market
    dm_grid = np.arange(0.0, 0.0801, 0.0005)
    prices  = np.array([price_frn(dm, el=0) for dm in dm_grid])
    implied_dm = dm_grid[np.argmin(np.abs(prices - MARKET_PRICE))]
    print(f'implied DM (M2)   = {implied_dm*1e4:.0f} bps  (market-implied credit spread)')

## Findings

- **Method 1 (loss view):** price ≈ par minus the EL principal haircut; base EL is
  small (~1.2% at rho 0.20) so price sits just below par. Falls with correlation
  (higher rho → higher E[BR loss] → lower price).
- **Method 2 (DM):** price falls ~linearly as DM widens; at DM = issue spread it
  prices ~par (sanity check).
- The two answer different questions: M1 = intrinsic (our loss view); M2 = market
  convention (spread demanded). Market price sits **between/against** these.
- Caveats: WAL & flat-SOFR assumptions; principal-only loss haircut (no timing);
  Method 1 double-counts risk (EL + credit spread) by construction; DM range is an
  assumption, not calibrated. **Market comparison pending a real BR price.**